In [2]:
import pandas as pd
import re, unicodedata

In [3]:
def validate_df(
    df: pd.DataFrame,
    *,
    not_null: list[str] = None,
    positive: list[str] = None,
    non_negative: list[str] = None,
    numeric: list[str] = None,
    bounds: dict[str, tuple] = None,
    key_cols: list[str] = None,
    df_name: str = "DataFrame"
):
    report = []

    not_null = not_null or []
    positive = positive or []
    non_negative = non_negative or []
    bounds = bounds or {}
    key_cols = key_cols or []
    numeric = numeric or []

    # Not-null Checks
    for col in not_null:
        n = df[col].isna().sum()
        if n > 0:
            report.append(f"{df_name}: {col} → {n} NaN")

    # > 0 Checks
    for col in positive:
        n = (df[col] <= 0).sum()
        if n > 0:
            report.append(f"{df_name}: {col} → {n} Werte ≤ 0")

    # ≥ 0 Checks
    for col in non_negative:
        n = (df[col] < 0).sum()
        if n > 0:
            report.append(f"{df_name}: {col} → {n} negative Werte")

    # Bounds
    for col, (lo, hi) in bounds.items():
        n = (~df[col].between(lo, hi) & df[col].notna()).sum()
        if n > 0:
            report.append(f"{df_name}: {col} → {n} Werte außerhalb [{lo}, {hi}]")

    # Uniqueness
    if key_cols:
        n = df.duplicated(subset=key_cols).sum()
        if n > 0:
            report.append(f"{df_name}: {n} Duplikate in {key_cols}")

    # Typen-Prüfung
    for col in numeric:
        if not pd.api.types.is_numeric_dtype(df[col]):
            n_coerce = pd.to_numeric(df[col], errors="coerce").isna().sum()
            report.append(
                f"{df_name}: {col} → kein numerischer Typ "
                f"(coerce→NaN: {n_coerce})"
            )

    # korrekte Länge
    if len(df) != 52:
        raise ValueError(f"Unerwartete Anzahl Zeilen: {len(df)} (erwartet: 52)")

    if report:
        print("Validierungsbericht")
        for r in report:
            print(" -", r)
    else:
        print(f"{df_name}: alle Checks bestanden")

    return report


In [5]:
def clean_and_sort(
    df: pd.DataFrame,
    *value_cols: str,
    source_col: str = "Name",
) -> pd.DataFrame:

    df = df.copy()

    def clean_name(raw: str) -> str:
        raw = unicodedata.normalize("NFKC", str(raw)).replace("\xa0", " ").strip()
    
        # Suffixe (am Ende)
        raw = re.sub(r"\s*,\s*krfr\.?\s*Stadt\s*$", "", raw, flags=re.IGNORECASE)
        raw = re.sub(r"\s*,\s*kreisfreie\s*Stadt\s*$", "", raw, flags=re.IGNORECASE)
        raw = re.sub(r"\s*,\s*Stadt\s*$", "", raw, flags=re.IGNORECASE)
        raw = re.sub(r"\s*,\s*Kreis\s*$", "", raw, flags=re.IGNORECASE)
    
        # Präfixe (am Anfang)
        raw = re.sub(r"^(Stadt|Kreis)\s+", "", raw, flags=re.IGNORECASE)
    
        return raw.strip()
    
    df["Name"] = df[source_col].apply(clean_name)
    
    # Städteregion Aachen -> Aachen
    mask = df["Name"].str.contains("aachen", case=False, na=False)
    df.loc[mask, "Name"] = "Aachen"

    # Spaltenauswahl: Name + value_cols oder alles außer Rohspalte
    base_cols = ["Name"]
    if value_cols:
        cols = base_cols + list(value_cols)
    else:
        cols = base_cols + [c for c in df.columns if c not in base_cols + [source_col]]

    cols = [c for c in cols if c in df.columns]
    cleaned_df = df[cols].reset_index(drop=True)


    return cleaned_df
